# Evaluate Fine-tuned Qwen2.5-0.5B

Runs the trained model on all 100 test puzzles (ranks 901–1000 from `24.csv`),
verifies each output arithmetically, and saves results to a JSON file.

**Run twice** — once for each model — to get the ablation comparison:
1. `./qwen05_finetuned` → `eval_results.json` (augmented, 1487 examples)
2. `./qwen05_clean_only_finetuned` → `eval_results_clean_only.json` (clean only, 446 examples)

## Configuration
**Change `MODEL_PATH` and `OUTPUT_FILE` for each run.**

In [1]:
# ── Change these two lines between the two evaluation runs ───────────────────
MODEL_PATH  = "./qwen05_clean_only_finetuned"   # or "./qwen05_finetuned" for the augmented model
OUTPUT_FILE = "eval_results_clean_only_sample_noseed.json"    # or "eval_results.json" for the augmented model

# ── Fixed settings (do not change) ───────────────────────────────────────────
CSV_PATH       = "24.csv"
RANK_START     = 901
RANK_END       = 1000
MAX_NEW_TOKENS = 300

## Prompt templates (must match training exactly)

In [2]:
SYSTEM_PROMPT = (
    "Use numbers and basic arithmetic operations (+, -, *, /) to obtain 24. "
    "Each step, you are only allowed to choose two of the remaining numbers to obtain a new number.\n"
    "Step 1: Start by considering possible operations for each pair of numbers.\n"
    "Step 2: Try a path (a pair of two numbers), see if the remaining numbers can possibly reach the goal 24. If not, backtrack and attempt another.\n"
    "Step 3: Branch out to try different orders of operations and combinations, evaluating each outcome.\n"
    "Step 4: If one path doesn't lead to a solution, backtrack and try alternative operations.\n\n"
)

IN_CONTEXT_HEADER = (
    "Here are some solved examples:\n\n"
    "Numbers: 4 4 6 8. Target: 24.\n"
    "Use each number exactly once with +, -, *, / to reach 24.\n"
    "Steps:\n"
    "4 + 8 = 12 (left: 4 6 12)\n"
    "6 - 4 = 2 (left: 2 12)\n"
    "2 * 12 = 24 (left: 24)\n"
    "Answer: (6 - 4) * (4 + 8) = 24\n\n"
    "Numbers: 1 4 8 8. Target: 24.\n"
    "Use each number exactly once with +, -, *, / to reach 24.\n"
    "Steps:\n"
    "8 / 4 = 2 (left: 1 2 8)\n"
    "1 + 2 = 3 (left: 3 8)\n"
    "3 * 8 = 24 (left: 24)\n"
    "Answer: (1 + 8 / 4) * 8 = 24\n\n"
    "Numbers: 5 5 5 9. Target: 24.\n"
    "Use each number exactly once with +, -, *, / to reach 24.\n"
    "Steps:\n"
    "5 + 5 = 10 (left: 5 9 10)\n"
    "10 + 5 = 15 (left: 9 15)\n"
    "15 + 9 = 24 (left: 24)\n"
    "Answer: ((5 + 5) + 5) + 9 = 24\n\n"
    "Numbers: 4 9 10 13. Target: 24.\n"
    "Use each number exactly once with +, -, *, / to reach 24.\n"
    "Steps:\n"
    "4 + 9 = 13 (left: 10 13 13)\n"
    "13 - 10 = 3 (left: 3 13)\n"
    "13 + 3 = 16 (left: 16)\n"
    "Backtrack. (to: 4 9 10 13).\n"
    "13 - 10 = 3 (left: 3 4 9)\n"
    "9 - 3 = 6 (left: 4 6)\n"
    "4 * 6 = 24 (left: 24)\n"
    "Answer: 4 * (9 - (13 - 10)) = 24\n\n"
    "Now solve this puzzle:\n"
)

def make_user_content(puzzle):
    return (
        SYSTEM_PROMPT
        + IN_CONTEXT_HEADER
        + f"Numbers: {puzzle}. Target: 24.\n"
        + "Use each number exactly once with +, -, *, / to reach 24.\n"
        + "Steps:"
    )

## Arithmetic verifier

In [3]:
import re
from fractions import Fraction

def verify_output(puzzle, response_text):
    puzzle_nums = [Fraction(x) for x in puzzle.strip().split()]
    lines = [l.strip() for l in response_text.strip().split('\n') if l.strip()]

    step_errors   = []
    steps_checked = 0
    steps_ok      = 0
    available     = list(puzzle_nums)
    answer_correct = False
    answer_msg     = "No Answer line found"

    for lt in lines:
        if lt.startswith('Backtrack.'):
            bm = re.search(r'\(to:\s*(.*?)\)', lt)
            if bm:
                raw_tokens = bm.group(1).strip().split()
                parsed = []
                for tok in raw_tokens:
                    try:
                        parsed.append(Fraction(tok))
                    except (ValueError, ZeroDivisionError):
                        pass
                available = parsed if parsed else list(puzzle_nums)
            else:
                available = list(puzzle_nums)
            continue

        if lt.startswith('Answer:'):
            am = re.match(r'Answer:\s*(.+?)\s*=\s*24', lt)
            if not am:
                answer_msg = f"Cannot parse: {lt!r}"
                continue
            expr = am.group(1)
            try:
                expr_f    = re.sub(r'\b(\d+)\b', r'Fraction(\1)', expr)
                result    = eval(expr_f, {"Fraction": Fraction, "__builtins__": {}})
                nums_used = sorted(int(x) for x in re.findall(r'\b\d+\b', expr))
                expected  = sorted(int(x) for x in puzzle.strip().split())
                if result == 24 and nums_used == expected:
                    answer_correct = True
                    answer_msg     = "Correct"
                elif result != 24:
                    answer_msg = f"Evaluates to {float(result):.4g}, not 24"
                else:
                    answer_msg = f"Wrong numbers used: {nums_used} vs {expected}"
            except Exception as e:
                answer_msg = f"Eval error: {e}"
            continue

        m = re.match(
            r'(-?[\d.]+)\s*([+\-*/])\s*(-?[\d.]+)\s*=\s*(-?[\d.]+)\s*\(left:\s*(.*?)\)',
            lt
        )
        if not m:
            continue

        steps_checked += 1
        a_s, op, b_s, c_s, left_s = m.groups()
        try:
            a, b, c = Fraction(a_s), Fraction(b_s), Fraction(c_s)
        except Exception:
            step_errors.append((lt, "Could not parse numbers"))
            continue

        ops_map    = {'+': a+b, '-': a-b, '*': a*b, '/': (a/b if b != 0 else None)}
        expected_c = ops_map.get(op)

        if expected_c is None:
            step_errors.append((lt, "Division by zero"))
        elif expected_c != c:
            step_errors.append((lt, f"Arithmetic error: {float(a)} {op} {float(b)} != {float(c)}"))
        else:
            steps_ok += 1
            try:
                available.remove(a)
                available.remove(b)
            except ValueError:
                step_errors.append((lt, f"Numbers not in pool: {float(a)}, {float(b)}"))
                steps_ok -= 1
                available.append(c)
                continue
            available.append(c)

    accuracy = steps_ok / steps_checked if steps_checked > 0 else 0.0
    return {
        "step_errors": step_errors, "answer_correct": answer_correct,
        "answer_msg": answer_msg, "arithmetic_accuracy": accuracy,
        "steps_checked": steps_checked,
    }

print("Verifier defined ✓")

Verifier defined ✓


## Load test puzzles

In [4]:
import csv

with open(CSV_PATH) as f:
    rows = list(csv.DictReader(f))

test_puzzles = [
    r["Puzzles"].strip()
    for r in rows
    if RANK_START <= int(r["Rank"]) <= RANK_END
]
print(f"Loaded {len(test_puzzles)} test puzzles (ranks {RANK_START}–{RANK_END})")

Loaded 100 test puzzles (ranks 901–1000)


## Load model

In [15]:
!pip show transformers

Name: transformers
Version: 5.9.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: G:\class codes\tot_preliminary_1\student\venv\Lib\site-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: 


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading {MODEL_PATH} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)
model.eval()
device = next(model.parameters()).device
print(f"Model loaded on {device} ✓")

g:\class codes\LLMtoSlm\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading ./qwen05_clean_only_finetuned ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:07<00:00, 41.24it/s]


Model loaded on cuda:0 ✓


## Run evaluation
This cell iterates over all 100 puzzles — expect ~5–15 minutes on GPU.

In [ ]:
per_puzzle      = []
no_answer_count = 0

for i, puzzle in enumerate(test_puzzles):
    user_content = make_user_content(puzzle)
    messages     = [{"role": "user", "content": user_content}]
    prompt_text  = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs    = tokenizer(prompt_text, return_tensors="pt").to(device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids  = out[0][input_len:]
    response = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    v = verify_output(puzzle, response)
    if v["answer_msg"] == "No Answer line found":
        no_answer_count += 1

    per_puzzle.append({
        "puzzle":              puzzle,
        "response":            response,
        "answer_correct":      v["answer_correct"],
        "answer_msg":          v["answer_msg"],
        "arithmetic_accuracy": v["arithmetic_accuracy"],
        "steps_checked":       v["steps_checked"],
        "step_error_count":    len(v["step_errors"]),
    })

    status = "✓" if v["answer_correct"] else "✗"
    print(f"  [{i+1:3d}/100] {status} {puzzle:<15} "
          f"arith={v['arithmetic_accuracy']:.0%}  step_errs={len(v['step_errors'])}")

print("\nEvaluation complete.")

  [  1/100] ✗ 4 5 6 10        arith=38%  step_errs=8
  [  2/100] ✗ 1 2 4 7         arith=100%  step_errs=0
  [  3/100] ✗ 2 5 8 11        arith=40%  step_errs=6
  [  4/100] ✗ 3 4 4 13        arith=75%  step_errs=1
  [  5/100] ✗ 6 7 8 9         arith=67%  step_errs=2
  [  6/100] ✓ 1 11 11 13      arith=100%  step_errs=0
  [  7/100] ✗ 1 8 10 11       arith=33%  step_errs=6
  [  8/100] ✗ 2 3 6 9         arith=36%  step_errs=9
  [  9/100] ✗ 1 3 5 9         arith=67%  step_errs=2
  [ 10/100] ✓ 3 3 7 12        arith=100%  step_errs=0
  [ 11/100] ✗ 4 5 7 9         arith=62%  step_errs=5
  [ 12/100] ✗ 1 2 8 13        arith=100%  step_errs=0
  [ 13/100] ✗ 4 6 6 9         arith=100%  step_errs=0
  [ 14/100] ✗ 1 4 4 8         arith=19%  step_errs=13
  [ 15/100] ✗ 1 5 10 11       arith=80%  step_errs=1
  [ 16/100] ✓ 3 4 6 11        arith=100%  step_errs=0
  [ 17/100] ✗ 2 4 8 9         arith=83%  step_errs=1
  [ 18/100] ✗ 1 4 5 13        arith=100%  step_errs=0
  [ 19/100] ✗ 2 2 7 12        arith=83

sampling version

In [ ]:
import torch

SEED = 42  # same seed for both model runs — isolates model as the only variable

per_puzzle      = []
no_answer_count = 0

for i, puzzle in enumerate(test_puzzles):
    user_content = make_user_content(puzzle)
    messages     = [{"role": "user", "content": user_content}]
    prompt_text  = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs    = tokenizer(prompt_text, return_tensors="pt").to(device)
    input_len = inputs["input_ids"].shape[1]

    torch.manual_seed(SEED + i)  # per-puzzle seed, but identical across both model runs

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.3,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids  = out[0][input_len:]
    response = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    v = verify_output(puzzle, response)
    if v["answer_msg"] == "No Answer line found":
        no_answer_count += 1

    per_puzzle.append({
        "puzzle":              puzzle,
        "response":            response,
        "answer_correct":      v["answer_correct"],
        "answer_msg":          v["answer_msg"],
        "arithmetic_accuracy": v["arithmetic_accuracy"],
        "steps_checked":       v["steps_checked"],
        "step_error_count":    len(v["step_errors"]),
    })

    status = "✓" if v["answer_correct"] else "✗"
    print(f"  [{i+1:3d}/100] {status} {puzzle:<15} "
          f"arith={v['arithmetic_accuracy']:.0%}  step_errs={len(v['step_errors'])}")

print("\nEvaluation complete.")

  [  1/100] ✓ 4 5 6 10        arith=100%  step_errs=0
  [  2/100] ✗ 1 2 4 7         arith=100%  step_errs=0
  [  3/100] ✗ 2 5 8 11        arith=75%  step_errs=2
  [  4/100] ✗ 3 4 4 13        arith=100%  step_errs=0
  [  5/100] ✗ 6 7 8 9         arith=67%  step_errs=2
  [  6/100] ✓ 1 11 11 13      arith=100%  step_errs=0
  [  7/100] ✗ 1 8 10 11       arith=27%  step_errs=11
  [  8/100] ✗ 2 3 6 9         arith=75%  step_errs=1
  [  9/100] ✗ 1 3 5 9         arith=100%  step_errs=0
  [ 10/100] ✗ 3 3 7 12        arith=100%  step_errs=0
  [ 11/100] ✗ 4 5 7 9         arith=75%  step_errs=3
  [ 12/100] ✗ 1 2 8 13        arith=100%  step_errs=0
  [ 13/100] ✗ 4 6 6 9         arith=19%  step_errs=13
  [ 14/100] ✗ 1 4 4 8         arith=67%  step_errs=2
  [ 15/100] ✗ 1 5 10 11       arith=42%  step_errs=7
  [ 16/100] ✓ 3 4 6 11        arith=100%  step_errs=0
  [ 17/100] ✓ 2 4 8 9         arith=100%  step_errs=0
  [ 18/100] ✗ 1 4 5 13        arith=25%  step_errs=12
  [ 19/100] ✗ 2 2 7 12        arit

sampling no seed

In [6]:
per_puzzle      = []
no_answer_count = 0

for i, puzzle in enumerate(test_puzzles):
    user_content = make_user_content(puzzle)
    messages     = [{"role": "user", "content": user_content}]
    prompt_text  = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs    = tokenizer(prompt_text, return_tensors="pt").to(device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.3,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids  = out[0][input_len:]
    response = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    v = verify_output(puzzle, response)
    if v["answer_msg"] == "No Answer line found":
        no_answer_count += 1

    per_puzzle.append({
        "puzzle":              puzzle,
        "response":            response,
        "answer_correct":      v["answer_correct"],
        "answer_msg":          v["answer_msg"],
        "arithmetic_accuracy": v["arithmetic_accuracy"],
        "steps_checked":       v["steps_checked"],
        "step_error_count":    len(v["step_errors"]),
    })

    status = "✓" if v["answer_correct"] else "✗"
    print(f"  [{i+1:3d}/100] {status} {puzzle:<15} "
          f"arith={v['arithmetic_accuracy']:.0%}  step_errs={len(v['step_errors'])}")

print("\nEvaluation complete.")

  [  1/100] ✗ 4 5 6 10        arith=36%  step_errs=9
  [  2/100] ✗ 1 2 4 7         arith=100%  step_errs=0
  [  3/100] ✗ 2 5 8 11        arith=21%  step_errs=11
  [  4/100] ✗ 3 4 4 13        arith=75%  step_errs=1
  [  5/100] ✗ 6 7 8 9         arith=75%  step_errs=1
  [  6/100] ✗ 1 11 11 13      arith=27%  step_errs=11
  [  7/100] ✗ 1 8 10 11       arith=33%  step_errs=8
  [  8/100] ✗ 2 3 6 9         arith=42%  step_errs=7
  [  9/100] ✗ 1 3 5 9         arith=67%  step_errs=2
  [ 10/100] ✓ 3 3 7 12        arith=100%  step_errs=0
  [ 11/100] ✗ 4 5 7 9         arith=67%  step_errs=3
  [ 12/100] ✗ 1 2 8 13        arith=100%  step_errs=0
  [ 13/100] ✗ 4 6 6 9         arith=70%  step_errs=3
  [ 14/100] ✗ 1 4 4 8         arith=80%  step_errs=1
  [ 15/100] ✗ 1 5 10 11       arith=64%  step_errs=4
  [ 16/100] ✗ 3 4 6 11        arith=50%  step_errs=3
  [ 17/100] ✗ 2 4 8 9         arith=75%  step_errs=1
  [ 18/100] ✗ 1 4 5 13        arith=71%  step_errs=2
  [ 19/100] ✓ 2 2 7 12        arith=100% 

## Summary and save results

In [7]:
import json

n        = len(per_puzzle)
correct  = sum(1 for r in per_puzzle if r["answer_correct"])
avg_acc  = sum(r["arithmetic_accuracy"] for r in per_puzzle) / n
total_se = sum(r["step_error_count"] for r in per_puzzle)

summary = {
    "model":              MODEL_PATH,
    "csv":                CSV_PATH,
    "rank_start":         RANK_START,
    "rank_end":           RANK_END,
    "n_puzzles":          n,
    "n_correct":          correct,
    "success_rate_pct":   round(correct / n * 100, 1),
    "no_answer_count":    no_answer_count,
    "avg_arithmetic_acc": round(avg_acc, 4),
    "total_step_errors":  total_se,
    "per_puzzle":         per_puzzle,
}

with open(OUTPUT_FILE, "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 60)
print(f"SUMMARY  ({n} puzzles, ranks {RANK_START}–{RANK_END})")
print("=" * 60)
print(f"  Model                  : {MODEL_PATH}")
print(f"  Correct answers (= 24) : {correct}/{n}  ({correct/n*100:.1f}%)")
print(f"  No-answer outputs      : {no_answer_count}")
print(f"  Avg arithmetic accuracy: {avg_acc:.0%}")
print(f"  Total step errors      : {total_se}")
print(f"\nResults saved -> {OUTPUT_FILE}")

SUMMARY  (100 puzzles, ranks 901–1000)
  Model                  : ./qwen05_clean_only_finetuned
  Correct answers (= 24) : 6/100  (6.0%)
  No-answer outputs      : 36
  Avg arithmetic accuracy: 68%
  Total step errors      : 349

Results saved -> eval_results_clean_only_sample_noseed.json


## Ablation comparison table
Run this cell **after both evaluation runs** are complete.

In [1]:
import json, os

results = []
for fpath, label, n_train in [
    ("eval_results_run1.json",            "Qwen (augmented, backtracking)", 1487),
    ("eval_results_clean_only.json", "Qwen (clean only)",              446),
]:
    if os.path.exists(fpath):
        with open(fpath) as f:
            d = json.load(f)
        results.append((label, n_train, d["success_rate_pct"], d["no_answer_count"]))
    else:
        results.append((label, n_train, "not yet run", "—"))

print(f"{'Model':<38} {'Train examples':>15} {'Success rate':>13} {'No-answer':>10}")
print("-" * 80)
for label, n_train, sr, na in results:
    sr_str = f"{sr}%" if isinstance(sr, float) else sr
    print(f"{label:<38} {n_train:>15} {sr_str:>13} {na:>10}")

Model                                   Train examples  Success rate  No-answer
--------------------------------------------------------------------------------
Qwen (augmented, backtracking)                    1487         19.0%         73
Qwen (clean only)                                  446         11.0%         30


take one from json one from txt

In [10]:
import json, os, re

def load_from_json(fpath):
    """Load metrics from a pre-aggregated eval_results*.json file."""
    with open(fpath) as f:
        d = json.load(f)
    return {
        "n_puzzles": d["n_puzzles"],
        "n_correct": d["n_correct"],
        "success_rate_pct": d["success_rate_pct"],
        "no_answer_count": d["no_answer_count"],
    }

def load_from_debug_txt(fpath):
    """Parse metrics out of a raw qwen debug-output .txt log.

    Each puzzle block looks like:
        PUZZLE: 4 5 6 10
        ...
        VERIFICATION:
          Arithmetic accuracy: 100% (9 steps checked)
          Answer line: ✗ Evaluates to 29, not 24
          No step errors ✓
    We just need to read out the "Answer line:" verdict per puzzle.
    """
    with open(fpath, encoding="utf-8", errors="replace") as f:
        text = f.read()

    answer_lines = re.findall(r"Answer line:\s*(.*)", text)

    n_puzzles = len(answer_lines)
    n_correct = sum(1 for a in answer_lines if a.strip().startswith("✓"))
    no_answer_count = sum(1 for a in answer_lines if "No Answer line found" in a)

    success_rate_pct = round(100 * n_correct / n_puzzles, 1) if n_puzzles else 0.0

    return {
        "n_puzzles": n_puzzles,
        "n_correct": n_correct,
        "success_rate_pct": success_rate_pct,
        "no_answer_count": no_answer_count,
    }

def load_metrics(fpath):
    if fpath.endswith(".json"):
        return load_from_json(fpath)
    elif fpath.endswith(".txt"):
        return load_from_debug_txt(fpath)
    else:
        raise ValueError(f"Unrecognized file type: {fpath}")


results = []
for fpath, label, n_train in [
    ("qwen ouput_debug_output_446 tree train_testrange901-1000.txt",                                                "Qwen (augmented, backtracking)", 1487),
    ("eval_results_clean_only.json",      "Qwen (clean only, 446)",          446),
]:
    if os.path.exists(fpath):
        m = load_metrics(fpath)
        results.append((label, n_train, m["success_rate_pct"], m["no_answer_count"], m["n_puzzles"]))
    else:
        results.append((label, n_train, "not yet run", "—", "—"))

print(f"{'Model':<38} {'Train examples':>15} {'Success rate':>13} {'No-answer':>10} {'N puzzles':>10}")
print("-" * 92)
for label, n_train, sr, na, n in results:
    sr_str = f"{sr}%" if isinstance(sr, float) else sr
    print(f"{label:<38} {n_train:>15} {sr_str:>13} {na:>10} {n:>10}")

Model                                   Train examples  Success rate  No-answer  N puzzles
--------------------------------------------------------------------------------------------
Qwen (augmented, backtracking)                    1487         24.0%         61        100
Qwen (clean only, 446)                             446         11.0%         30        100
